Step 1: Bootstrap & installation

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
from google.colab import userdata, drive
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')

Step 2: Integration test of react agent

In [ ]:
from astra_swarm.router import classify_alert
from astra_swarm.react_agent import react_triage
from astra_swarm.cassette import cassette
import json
from pathlib import Path

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/10_alerts.json").read_text()
)

results = []
with cassette("day_a_router_agent_10alerts"):
    for i, a in enumerate(alerts, 1):
        routing = classify_alert(a)
        try:
            investigation = react_triage(a, routing)
        except RuntimeError as e:
            print(f"[{i:2}] {routing.alert_class.value:<14} → FAILED: {e}")
            continue
        results.append({"routing": routing, "investigation": investigation})
        print(f"[{i:2}] {routing.alert_class.value:<14} → "
              f"sev={investigation.severity.value:<10} "
              f"techs={len(investigation.attack_techniques):<2} "
              f"rounds={investigation.rounds_used}")